In [40]:
!pip install tdqm


In [41]:
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from tqdm import tqdm
import string



In [42]:
file = '/content/Emotion_classify_Data.csv'
df=pd.read_csv(file)
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [43]:
#Data_Tokenazing---------------------------------------------

nltk.download('punkt_tab')
nltk.download('stopwords')
tqdm.pandas()

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_emotion_text(text):

    if pd.isna(text):
      return[]

    try:

      text = text.lower()
      tokens = word_tokenize(text)


      clean_tokens = [token for token in tokens if token.isalpha()]
      final_tokens = [token for token in clean_tokens if token not in stop_words]
      return final_tokens


    except Exception as e:
       print(f"خطا در پردازش: {text[:50]}... - {e}")
       return []

#Data_Preprocessing---------------------------------------------
text_columns='Comment'
df['tokens'] = df[text_columns].progress_apply(preprocess_emotion_text)
df['cleaned_text'] = df['tokens'].apply(lambda x: ' '.join(x))
df['token_count'] = df['tokens'].apply(len)

df.head()

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
100%|██████████| 5937/5937 [00:01<00:00, 4814.90it/s]


,Comment,Emotion,tokens,cleaned_text,token_count
0,i seriously hate one subject to death but now ...,fear,"[seriously, hate, one, subject, death, feel, r...",seriously hate one subject death feel reluctan...,8
1,im so full of life i feel appalled,anger,"[im, full, life, feel, appalled]",im full life feel appalled,5
2,i sit here to write i start to dig out my feel...,fear,"[sit, write, start, dig, feelings, think, afra...",sit write start dig feelings think afraid acce...,11
3,ive been really angry with r and i feel like a...,joy,"[ive, really, angry, r, feel, like, idiot, tru...",ive really angry r feel like idiot trusting fi...,10
4,i feel suspicious if there is no one outside l...,fear,"[feel, suspicious, one, outside, like, rapture...",feel suspicious one outside like rapture happe...,8


In [45]:
from sklearn.model_selection import train_test_split

#Train/Test_Split(train=0.70,test=0.15,validiation=0.15)----------------------------------

#Features---------
X = df['cleaned_text']

#Lables---------
y = df['Emotion']

#(train=0.70,test=0.15)------------
X_train,X_test,y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

#validiation=0.15------------------
X_validation,X_train_new,y_validation, y_train_new = train_test_split(
  X_train, y_train, test_size=0.1756, random_state=42, stratify=y_train)



In [46]:

#Frequency_word---------------------------

from collections import Counter

for emotion in df['Emotion'].unique():
    texts = df[df['Emotion']==emotion]['cleaned_text']
    all_words = ' '.join(texts.dropna()).split()
    top_words = Counter(all_words).most_common(7)


    print(f"{emotion}:")
    for word, count in top_words:
        print(f"__  {word}:{count} بار")

fear:
__  feel:1212 بار
__  feeling:742 بار
__  im:322 بار
__  like:258 بار
__  little:149 بار
__  know:117 بار
__  bit:117 بار
anger:
__  feel:1355 بار
__  feeling:669 بار
__  like:355 بار
__  im:321 بار
__  get:115 بار
__  really:113 بار
__  time:109 بار
joy:
__  feel:1480 بار
__  feeling:553 بار
__  like:379 بار
__  im:300 بار
__  really:110 بار
__  time:102 بار
__  get:99 بار


In [48]:

#common_words_to_remove------------------------

common_words_to_remove = {'feel', 'like', 'feeling','im',
                          'really','time','get','know',
                          'little','want','people','still',
                          'would','even','way','think',
                          'one','things','could','bit',
                          'dont','cold'}


df['filtered_tokens'] = df['tokens'].apply(
    lambda x: [word for word in x if word not in common_words_to_remove]
    if isinstance(x, list) else []

)

df['filtered_cleaned_text'] = df['filtered_tokens'].apply(lambda x: ' '.join(x))



In [49]:

#most_common_top_words-----------------------------------------------

print(" کلمات پرتکرار بعد از حذف کلمات مشترک:")

for emotion in df['Emotion'].unique():
    texts = df[df['Emotion'] == emotion]['filtered_cleaned_text']
    all_words = ' '.join(texts.dropna()).split()
    top_words = Counter(all_words).most_common(2)

    print(f"\n{emotion}:")
    for word, count in top_words:
        print(f"  {word}: {count} بار")

 کلمات پرتکرار بعد از حذف کلمات مشترک:

fear:
  strange: 78 بار
  nervous: 75 بار

anger:
  angry: 79 بار
  irritable: 63 بار

joy:
  make: 78 بار
  good: 76 بار


In [50]:
#feature(Word)_extraction----------------------------------

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(max_features=1500)
X = vectorizer.fit_transform(df['filtered_cleaned_text'])
feature_names = vectorizer.get_feature_names_out()


for i, feature in enumerate(feature_names[:30]):
    print(f"  {i+1}. '{feature}'")

  1. 'ability'
  2. 'able'
  3. 'absolutely'
  4. 'accept'
  5. 'acceptable'
  6. 'accepted'
  7. 'accident'
  8. 'accomplished'
  9. 'across'
  10. 'act'
  11. 'actions'
  12. 'activity'
  13. 'acts'
  14. 'actual'
  15. 'actually'
  16. 'add'
  17. 'added'
  18. 'admit'
  19. 'adult'
  20. 'advantage'
  21. 'adventurous'
  22. 'advice'
  23. 'afraid'
  24. 'afternoon'
  25. 'afterwards'
  26. 'age'
  27. 'aggravated'
  28. 'agitated'
  29. 'ago'
  30. 'ahead'


In [51]:
#feature(Bigarms)_extraction----------------------------------

from sklearn.feature_extraction.text import TfidfVectorizer


sentences = df['filtered_cleaned_text'].dropna()
bigram_vectorizer = TfidfVectorizer(
    ngram_range=(2, 2),
    max_features=1500
)


X = bigram_vectorizer.fit_transform(sentences)
bigram_features = bigram_vectorizer.get_feature_names_out()


for i, bigram in enumerate(bigram_features[:30]):
    print(f"  {i+1}. '{bigram}'")

  1. 'able concentrate'
  2. 'able find'
  3. 'able help'
  4. 'able leave'
  5. 'able share'
  6. 'able speak'
  7. 'absolutely terrified'
  8. 'accepted allowed'
  9. 'accident car'
  10. 'act positive'
  11. 'actually eager'
  12. 'actually frightened'
  13. 'actually less'
  14. 'actually somewhat'
  15. 'actually starting'
  16. 'actually stop'
  17. 'admit hesitant'
  18. 'admit intimidated'
  19. 'admit skeptical'
  20. 'afraid going'
  21. 'afraid talk'
  22. 'ages ago'
  23. 'agitated day'
  24. 'agitated lately'
  25. 'agitated peckish'
  26. 'agitated right'
  27. 'agitated something'
  28. 'agitated today'
  29. 'ahead us'
  30. 'almost angry'


In [52]:
# Feature_Concatenation

from sklearn.feature_extraction.text import CountVectorizer
import scipy
import numpy as np

# 1-----------
unigram_vectorizer = CountVectorizer(max_features=1500)
X_unigrams = unigram_vectorizer.fit_transform(df['filtered_cleaned_text'])


#2--------------------
bigram_vectorizer = CountVectorizer(ngram_range=(2, 2),
                                    max_features=1500)
X_bigrams = bigram_vectorizer.fit_transform(df['filtered_cleaned_text'])


#3--------------------
X_combined = scipy.sparse.hstack([X_unigrams, X_bigrams])

print(X_combined.shape)

(5937, 3000)


In [54]:
# X_split----------------------------------------------

vectorizer = TfidfVectorizer(max_features=1500, ngram_range=(1, 2))

X_train_features = vectorizer.fit_transform(X_train_new)
X_validation_features = vectorizer.transform(X_validation)
X_test_features = vectorizer.transform(X_test)


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
#Logistic_Regression

model= LogisticRegression(max_iter=1000)
model.fit(X_train_features, y_train_new)

y_val_pred = model.predict(X_validation_features)
val_accuracy = accuracy_score(y_validation,y_val_pred)



y_test_pred = model.predict(X_test_features)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Accuracy on Validation Set:", val_accuracy)
print("Accuracy on Test Set:", test_accuracy)
print(classification_report(y_test, y_test_pred))

In [59]:
#Naive_Bayes

from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB()
clf.fit(X_train_features,y_train_new)


MultinomialNB_val_accuracy =  clf.score(X_validation_features, y_validation)
MultinomialNB_test_accuracy = clf.score(X_test_features, y_test)

print("Accuracy on Validation Set:", MultinomialNB_val_accuracy)
print("Accuracy on Test Set:", MultinomialNB_test_accuracy)
print(classification_report(y_test, y_test_pred))

Accuracy on Validation Set: 0.8107718201490743
Accuracy on Test Set: 0.7934904601571269
              precision    recall  f1-score   support

       anger       0.84      0.84      0.84       300
        fear       0.80      0.83      0.81       291
         joy       0.81      0.78      0.79       300

    accuracy                           0.81       891
   macro avg       0.81      0.81      0.81       891
weighted avg       0.81      0.81      0.81       891



In [61]:
#Svm

from sklearn import svm
clf = svm.SVC()
clf.fit(X_train_features,y_train_new)

svm_val_accuracy =  clf.score(X_validation_features, y_validation)
svm_test_accuracy = clf.score(X_test_features, y_test)

print("Accuracy on Validation Set:", svm_val_accuracy)
print("Accuracy on Test Set:", svm_test_accuracy)


Accuracy on Validation Set: 0.8175042077422457
Accuracy on Test Set: 0.8080808080808081
